# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library, leveraging Croissant schema metadata and referencing all data entities by their unique `@id`.

### Dataset Source
The dataset is described by a Croissant schema available at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (not as dict)
metadata = dataset.metadata

# Print dataset metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant schemas organize tabular or structured data by `recordSet` entities (each uniquely identified by an `@id`). Each `recordSet` references fields and columns (also by `@id`). Let's list all record sets and their fields by `@id`.

In [ ]:
# List all record sets (by @id)

record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")

# For each record set, list fields (by @id)
for rs in record_sets:
    print(f"\nRecord Set: {rs.id}")
    print(f"Fields:")
    for field in rs.fields:
        print(f"  - @id: {field.id} | name: {getattr(field, 'name', 'N/A')} | dataType: {getattr(field, 'data_type', 'N/A')}")

# Display a sample of records for each record set
for rs in record_sets:
    print(f"\nSample records from Record Set (@id): {rs.id}")
    for x in dataset.records(record_set=rs.id):
        print(x)
        break  # Show only first record for brevity

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Fields and columns are referenced by their `@id`. For demonstration, the notebook will extract all available record sets, building a mapping `{record_set @id: DataFrame}`.

In [ ]:
# Build mapping: {record_set @id: DataFrame}
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns for each record set
for rs_id, df in dataframes.items():
    print(f"Record Set (@id): {rs_id} columns:")
    print(df.columns.tolist())
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using fields referenced by `@id`. We'll select a numeric field and a group field from a chosen record set above for demonstration.

For illustration, let's select the first available record set containing a numeric column, filter by threshold, normalize, and group.

In [ ]:
# Identify a record set with numeric field
selected_rs_id = None
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    # Find float/integer fields
    num_fields = [field for field in rs.fields if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']]
    # Find possible grouping categorical fields
    cat_fields = [field for field in rs.fields if getattr(field, 'data_type', '').lower() in ['text', 'string']]
    if num_fields:
        selected_rs_id = rs.id
        numeric_field_id = num_fields[0].id
        group_field_id = cat_fields[0].id if cat_fields else None
        break

# Confirm
print(f"Selected record set: {selected_rs_id}")
print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}")

df = dataframes[selected_rs_id]
if numeric_field_id in df.columns:
    threshold = np.nanmean(df[numeric_field_id])
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field (if available)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize numeric field distributions and relationships.
The visualizations reference fields and columns by `@id` for clarity and reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # Boxplot by group_field (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
This notebook demonstrated systematic exploration of a Croissant-based dataset using `mlcroissant`, referencing all entities by `@id` for reproducibility. You inspected record set structure, loaded tabular data, performed basic filtering and normalization, grouped by categories, and visualized the dataset.

Further analyses are enabled by the complete mapping from Croissant schema to pandas DataFrames.